# Statistical Significance Tests

Pairwise Wilcoxon signed-rank tests across 20 seeds for all 5 models.

- Primary metric: AUC
- Secondary: FAR, Event Sensitivity
- Output: p-value matrix table + LaTeX code

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings('ignore')

RESULTS_DIRS = {
    'PSD+LDA':        r'D:\seizure_results\psd_lda\results.json',
    '1D-CNN':         r'D:\seizure_results\1dcnn\results.json',
    'EEGNet':         r'D:\seizure_results\eegnet\results.json',
    'TCN':            r'D:\seizure_results\tcn\results.json',
    'EEG-Conformer':  r'D:\seizure_results\eeg_conformer\results.json',
}

# Metric key in results.json → display name
METRICS = {
    'test_auc':          'AUC',
    'far':               'FAR (/h)',
    'event_sensitivity': 'Event Sensitivity',
    'test_sen':          'Sensitivity (win)',
    'test_f1':           'F1',
}

print('Loading results...')

In [ ]:
# Load all results into a dict: model_name -> list of per-seed dicts
data = {}
for model, path in RESULTS_DIRS.items():
    try:
        with open(path) as f:
            data[model] = json.load(f)
        print(f'  {model}: {len(data[model])} seeds loaded')
    except FileNotFoundError:
        print(f'  {model}: NOT FOUND at {path}')

models = list(data.keys())
n_seeds = len(next(iter(data.values())))
print(f'\n{len(models)} models, {n_seeds} seeds each')

In [ ]:
# ── Summary Table (mean ± std for all models × metrics) ──
rows = []
for model in models:
    row = {'Model': model}
    for key, label in METRICS.items():
        vals = [r[key] for r in data[model] if key in r]
        if vals:
            row[label] = f'{np.mean(vals):.4f} ± {np.std(vals, ddof=1):.4f}'
        else:
            row[label] = 'N/A'
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Model')
print('\n=== Summary Table (mean ± std, n=20 seeds) ===')
display(summary_df)

In [ ]:
# ── Pairwise Wilcoxon Signed-Rank Tests ──
# For each metric, compute n×n p-value matrix.
# wilcoxon(a, b) tests H0: a and b come from same distribution.
# We test per-seed paired differences.

def pairwise_wilcoxon(data, models, metric_key):
    """
    Returns (n_models × n_models) DataFrame of p-values.
    p[i,j] = p-value for wilcoxon(model_i_scores, model_j_scores).
    Diagonal is NaN.
    """
    n = len(models)
    pmat = np.full((n, n), np.nan)
    scores = {}
    for m in models:
        scores[m] = np.array([r[metric_key] for r in data[m] if metric_key in r])

    for i, mi in enumerate(models):
        for j, mj in enumerate(models):
            if i == j:
                continue
            a, b = scores[mi], scores[mj]
            min_len = min(len(a), len(b))
            try:
                _, p = wilcoxon(a[:min_len], b[:min_len], alternative='two-sided')
            except ValueError:
                p = 1.0
            pmat[i, j] = p

    return pd.DataFrame(pmat, index=models, columns=models)

pval_tables = {}
for key, label in METRICS.items():
    pval_tables[label] = pairwise_wilcoxon(data, models, key)
    print(f'\n=== {label} — Pairwise Wilcoxon p-values ===')
    display(pval_tables[label].round(4))

In [ ]:
# ── Heatmap: AUC p-value matrix ──
import matplotlib.ticker as ticker

def plot_pval_heatmap(pmat_df, title, alpha=0.05, save_path=None):
    n = len(pmat_df)
    vals = pmat_df.values.copy()

    fig, ax = plt.subplots(figsize=(7, 5.5))
    # Use log scale for colours so small p-values pop
    log_vals = np.where(np.isnan(vals), np.nan, -np.log10(np.clip(vals, 1e-10, 1)))

    im = ax.imshow(log_vals, cmap='RdYlGn', aspect='auto', vmin=0, vmax=4)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('-log₁₀(p)', fontsize=10)
    cbar.ax.axhline(-np.log10(alpha), color='black', linewidth=1.5, linestyle='--')

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(pmat_df.columns, rotation=30, ha='right', fontsize=10)
    ax.set_yticklabels(pmat_df.index, fontsize=10)

    # Annotate cells
    for i in range(n):
        for j in range(n):
            if np.isnan(vals[i, j]):
                continue
            p = vals[i, j]
            stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
            txt = f'{p:.3f}\n{stars}'
            color = 'white' if log_vals[i, j] > 2 else 'black'
            ax.text(j, i, txt, ha='center', va='center', fontsize=8, color=color)

    ax.set_title(f'{title}\n(* p<0.05, ** p<0.01, *** p<0.001)', fontsize=11)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()

import os
os.makedirs(r'D:\seizure_results\figures', exist_ok=True)

plot_pval_heatmap(
    pval_tables['AUC'],
    'Pairwise Wilcoxon p-values — AUC (20 seeds)',
    save_path=r'D:\seizure_results\figures\wilcoxon_auc.png'
)

In [ ]:
# FAR heatmap (lower FAR is better)
plot_pval_heatmap(
    pval_tables['FAR (/h)'],
    'Pairwise Wilcoxon p-values — FAR (/h) (20 seeds)',
    save_path=r'D:\seizure_results\figures\wilcoxon_far.png'
)

In [ ]:
# ── Effect size: Cohen's d (paired) for AUC ──
# Useful to report alongside p-values in the paper.

def cohens_d_paired(a, b):
    diff = np.array(a) - np.array(b)
    return diff.mean() / (diff.std(ddof=1) + 1e-12)

auc_key = 'test_auc'
auc_scores = {m: np.array([r[auc_key] for r in data[m]]) for m in models}

print('\nCohen\'s d (paired) for AUC — row model minus column model')
d_mat = pd.DataFrame(index=models, columns=models, dtype=float)
for mi in models:
    for mj in models:
        if mi == mj:
            d_mat.loc[mi, mj] = np.nan
        else:
            n = min(len(auc_scores[mi]), len(auc_scores[mj]))
            d_mat.loc[mi, mj] = cohens_d_paired(auc_scores[mi][:n], auc_scores[mj][:n])
display(d_mat.round(3))

In [ ]:
# ── LaTeX output for paper ──
# Table 1: Summary (mean ± std)

col_order = ['AUC', 'Sensitivity (win)', 'Event Sensitivity', 'FAR (/h)', 'F1']
col_order = [c for c in col_order if c in summary_df.columns]

print('% ── LaTeX Table: Cross-Patient Results (mean ± std) ──')
print(r'\begin{table}[htbp]')
print(r'\centering')
print(r'\caption{Cross-patient (PI) results on CHB-MIT (mean $\pm$ std, $n=20$ seeds)}')
print(r'\label{tab:pi_results}')
ncols = len(col_order)
print(r'\begin{tabular}{l' + 'c' * ncols + r'}')
print(r'\toprule')
header = ' & '.join(['Model'] + col_order) + r' \\'
print(header)
print(r'\midrule')
for model in models:
    cells = [model]
    for c in col_order:
        cells.append(summary_df.loc[model, c] if c in summary_df.columns else '-')
    print(' & '.join(cells) + r' \\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

In [ ]:
# ── LaTeX output: AUC p-value table ──
print('% ── LaTeX Table: Wilcoxon p-values (AUC) ──')
pmat = pval_tables['AUC']
n = len(models)
print(r'\begin{table}[htbp]')
print(r'\centering')
print(r'\caption{Pairwise Wilcoxon signed-rank test $p$-values for AUC across 20 seeds. * $p<0.05$, ** $p<0.01$, *** $p<0.001$}')
print(r'\label{tab:wilcoxon}')
print(r'\begin{tabular}{l' + 'c' * n + r'}')
print(r'\toprule')
short = [m.replace('EEG-Conformer', 'EEG-Conf.').replace('PSD+LDA', 'LDA') for m in models]
print(' & '.join([''] + short) + r' \\')
print(r'\midrule')
for i, mi in enumerate(models):
    cells = [short[i]]
    for j, mj in enumerate(models):
        if i == j:
            cells.append('—')
        else:
            p = pmat.iloc[i, j]
            stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
            cells.append(f'{p:.3f}{stars}')
    print(' & '.join(cells) + r' \\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')

In [ ]:
import json, numpy as np
from itertools import combinations

models = ['PSD+LDA', '1D-CNN', 'EEGNet', 'TCN', 'EEG-Conformer']

json_paths = {
    'PSD+LDA':       r'D:\seizure_results\psd_lda\results.json',
    '1D-CNN':        r'D:\seizure_results\1dcnn\results.json',
    'EEGNet':        r'D:\seizure_results\eegnet\results.json',
    'TCN':           r'D:\seizure_results\tcn\results.json',
    'EEG-Conformer': r'D:\seizure_results\eeg_conformer\results.json',
}

model_aucs = {}
for m, path in json_paths.items():
    with open(path) as f:
        data = json.load(f)
    if isinstance(data, list):
        model_aucs[m] = [r['test_auc'] for r in data]
    elif isinstance(data, dict) and 'results' in data:
        model_aucs[m] = [r['test_auc'] for r in data['results']]
    else:
        model_aucs[m] = [v['test_auc'] for v in data.values()]
    print(f"{m}: {len(model_aucs[m])} seeds, mean={np.mean(model_aucs[m]):.4f}")

np.random.seed(0)
N_BOOT = 10000

print("\n=== Bootstrap 95% CI for ΔAUC (row − column) ===")
ci_matrix = {}
for m1 in models:
    ci_matrix[m1] = {}
    for m2 in models:
        if m1 == m2:
            ci_matrix[m1][m2] = None
            continue
        diffs = np.array(model_aucs[m1]) - np.array(model_aucs[m2])
        boot_means = [np.mean(diffs[np.random.randint(0, 20, 20)]) for _ in range(N_BOOT)]
        lo, hi = np.percentile(boot_means, [2.5, 97.5])
        ci_matrix[m1][m2] = (lo, hi)

print("\n=== LaTeX lower triangle (paste into Table 3) ===")
for i, m1 in enumerate(models):
    for j, m2 in enumerate(models):
        if j < i:
            lo, hi = ci_matrix[m1][m2]
            sig = "  ← EXCLUDES ZERO" if (lo > 0 or hi < 0) else ""
            bold = lo > 0 or hi < 0
            cell = f"[{lo:+.3f}, {hi:+.3f}]"
            if bold:
                cell = f"\\textbf{{{cell}}}"
            print(f"  {m1:<20} vs {m2:<20}: {cell}{sig}")